# IEP-1002 Day 2 — Step 2: 청킹 핵심 로직


## Step 2 개요

Step 1 결과를 분석하면 헤딩이 150 → 68개로 줄었으며 두 가지 문제가 확인됩니다.

**문제 1 — 본문 헤딩 미탐지**  
목차 텍스트 끝에 `---` 대시가 붙어 있어(`2.1 관측된 변화 ---`) 본문 헤딩과 텍스트가 달랐고,
교차 검증이 "목차 전용"으로 오판해 제거했습니다.  
실제로는 본문 페이지에서 `숫자점숫자` / `숫자점숫자점숫자` 패턴 헤딩이 정상 탐지되지 않은 것입니다.  
→ **이 단계에서 `ipcc_pages.json`을 직접 재스캔해 누락 헤딩을 보완합니다.**

**문제 2 — 수동 오탐 3개 미제거 (p.33, p.99, p.100)**  
Step 1에서 `MANUAL_REMOVE`가 비어 있었습니다.  
→ **이 단계에서 직접 필터링합니다.**

### 이 노트북의 처리 순서
1. `ipcc_pages.json` 재스캔으로 누락 헤딩 보완 → 최종 헤딩 리스트 확정  
2. 오탐 3개 제거  
3. 헤딩 경계 슬라이딩으로 청크 추출  
4. Fallback 적용 (최소 100자 병합 / 최대 2000자 분할)  
5. 청크 메타데이터 부여 + JSON 저장  


### 셀 1 — 데이터 로드
- 확인할 것: pages_data 188개, headings_clean 68개가 정상 로드되는지.


In [ ]:
import json, re
from pathlib import Path
from collections import Counter

OUTPUT_DIR = "/content/drive/MyDrive/IPCC_data/parsed"

with open(f"{OUTPUT_DIR}/ipcc_pages.json", encoding="utf-8") as f:
    pages_data = json.load(f)               # [{page_num, text, char_count}, ...]

with open(f"{OUTPUT_DIR}/ipcc_headings_clean.json", encoding="utf-8") as f:
    headings_clean = json.load(f)           # [{page, pattern, text}, ...] — 68개

# pages_data를 dict로 변환 (page_num → text)
page_text = {p["page_num"]: p["text"] for p in pages_data}

print(f"pages_data   : {len(pages_data)}페이지")
print(f"headings_clean: {len(headings_clean)}개")
print()

# Step 1 최종 분포 재확인
cnt = Counter(h["pattern"] for h in headings_clean)
print("Step 1 산출 헤딩 패턴:")
for p, c in sorted(cnt.items(), key=lambda x: -x[1]):
    print(f"  {p:20s}: {c}개")


### 셀 2 — 누락 헤딩 진단

목차에 있던 `숫자점숫자` / `숫자점숫자점숫자` 헤딩이 본문에서 몇 개나 탐지되는지 확인합니다.  
탐지 수가 목차 수와 크게 차이나면 본문에서 헤딩 텍스트 형태가 다른 것입니다.
- 확인할 것: `본문 탐지` 수가 목차 수보다 적은 헤딩 유형을 파악하세요.


In [ ]:
# ── 누락 헤딩 진단: 목차 헤딩 텍스트를 본문에서 직접 재탐지 ────────

# 목차 원본 (Day 1 ipcc_headings.json)에서 목차 페이지(p.13~14) 헤딩을 읽어옴
with open(f"{OUTPUT_DIR}/ipcc_headings.json", encoding="utf-8") as f:
    headings_raw = json.load(f)

toc_headings = [h for h in headings_raw if h["page"] in (13, 14)]

# 목차 헤딩 텍스트에서 대시 제거해 클린 제목 추출
def clean_toc_title(text: str) -> str:
    """목차 텍스트의 trailing 대시와 공백 제거"""
    return re.sub(r"[-\s]+$", "", text).strip()

# 본문 전체에서 각 클린 제목이 등장하는지 확인
BODY_START = 15  # 목차(p.13~14) 이후 본문 시작 페이지

found, missing = [], []
for h in toc_headings:
    title = clean_toc_title(h["text"])
    match_pages = [
        pnum for pnum, text in page_text.items()
        if pnum >= BODY_START and title in text
    ]
    if match_pages:
        found.append({"title": title, "pattern": h["pattern"], "pages": match_pages})
    else:
        missing.append({"title": title, "pattern": h["pattern"]})

print(f"목차 헤딩 총 {len(toc_headings)}개 → 본문 탐지: {len(found)}개 / 미탐지: {len(missing)}개")
print()
if missing:
    print("── 본문에서 찾지 못한 목차 헤딩 ──────────────────────")
    for m in missing:
        print(f"  [{m['pattern']:18s}]  {m['title'][:70]}")


### 셀 3 — 누락 헤딩 보완 (본문 재스캔)

목차 클린 제목으로 본문을 직접 스캔해 헤딩이 등장하는 페이지를 찾고, Step 1 결과에 추가합니다.  
중복 (같은 page + text)은 자동으로 제거합니다.
- 확인할 것: 보완 후 헤딩 수가 합리적인지 (100개 이상이면 정상). 동일 헤딩이 연속 2페이지에서 탐지되는 경우(2단 레이아웃)는 첫 번째만 유지합니다.


In [ ]:
# ── 누락 헤딩 보완 ──────────────────────────────────────────

# 본문에서 각 클린 제목의 등장 페이지를 탐색
supplemental = []
for item in found:
    title  = item["title"]
    pattern = item["pattern"]
    # Step 1 결과에 이미 포함된 페이지는 제외
    existing_pages = {
        h["page"] for h in headings_clean
        if h["text"] == title or h["text"].startswith(title)
    }
    for pnum in sorted(item["pages"]):
        if pnum not in existing_pages:
            supplemental.append({"page": pnum, "pattern": pattern, "text": title})

print(f"보완 후보: {len(supplemental)}개")

# Step 1 결과 + 보완 헤딩 합치기
headings_all = headings_clean + supplemental

# 중복 (page, text) 제거
seen, deduped = set(), []
for h in headings_all:
    key = (h["page"], h["text"])
    if key not in seen:
        seen.add(key)
        deduped.append(h)

# 동일 헤딩이 연속 페이지에 있으면 첫 번째만 유지 (2단 레이아웃 중복)
def remove_consecutive_dupes(headings: list) -> list:
    """같은 텍스트가 페이지 n, n+1에 연속 등장하면 n만 유지"""
    result = []
    for h in headings:
        if result and result[-1]["text"] == h["text"] and h["page"] - result[-1]["page"] <= 1:
            continue  # 연속 중복 제거
        result.append(h)
    return result

# 페이지 순 정렬 후 연속 중복 제거
deduped.sort(key=lambda h: (h["page"], h["text"]))
headings_final = remove_consecutive_dupes(deduped)

print(f"합치기 전: {len(headings_clean)}개 (Step 1) + {len(supplemental)}개 (보완)")
print(f"중복 제거: {len(deduped) - len(headings_final)}건")
print(f"최종 헤딩: {len(headings_final)}개")
print()

cnt2 = Counter(h["pattern"] for h in headings_final)
print("보완 후 패턴 분포:")
for p, c in sorted(cnt2.items(), key=lambda x: -x[1]):
    print(f"  {p:20s}: {c}개")


### 셀 4 — 수동 오탐 3개 제거

Step 1 셀 4 출력에서 확인된 오탐:
- p.33: `7.0 및 SSP5-8.5에 표시된다...` → 그림 범례 (본문 참조 줄)
- p.99: `2.5 SSP2-4.5` → 그림 축 레이블
- p.100: `Box1에서 제공된다.` → 본문 내 Box 참조

- 확인할 것: 세 항목이 모두 제거됐는지, 제거 후 헤딩 수를 확인하세요.


In [ ]:
# ── 수동 오탐 제거 ──────────────────────────────────────────
# (page, text_startswith) 형태로 지정
MANUAL_REMOVE = [
    (33,  "7.0 및 SSP5-8.5"),   # 그림 범례 — 숫자점숫자 오탐
    (99,  "2.5 SSP2-4.5"),      # 그림 축 레이블 — 숫자점숫자 오탐
    (100, "Box1에서 제공된다"),   # 본문 Box 참조 — Box 오탐
]

def is_manual_fp(h: dict) -> bool:
    for page, prefix in MANUAL_REMOVE:
        if h["page"] == page and h["text"].startswith(prefix):
            return True
    return False

removed = [h for h in headings_final if is_manual_fp(h)]
headings_final = [h for h in headings_final if not is_manual_fp(h)]

print(f"수동 오탐 제거: {len(removed)}개")
for h in removed:
    print(f"  p.{h['page']:3d}  [{h['pattern']:18s}]  {h['text'][:70]}")
print()
print(f"최종 청킹용 헤딩: {len(headings_final)}개")


### 셀 5 — 청킹 함수 정의

헤딩 ~ 다음 헤딩 사이 텍스트를 청크 단위로 추출합니다.  
Fallback 기준:
- **최소 100자 미만** → 다음 청크와 병합
- **최대 2000자 초과** → 문장 경계(마침표 + 개행)에서 고정 길이 분할

각 청크에는 `heading`, `page_start`, `page_end`, `char_count`, `fallback` 메타데이터를 포함합니다.
- 확인할 것: 함수 정의 셀이므로 오류 없이 실행만 되면 됩니다.


In [ ]:
# ── 청킹 함수 정의 ──────────────────────────────────────────

MIN_CHARS  = 100    # 이보다 짧으면 다음 청크와 병합
MAX_CHARS  = 2000   # 이보다 길면 문장 경계에서 분할


def extract_raw_chunks(headings: list, page_text: dict) -> list:
    """
    헤딩 리스트와 페이지 텍스트 dict를 받아
    헤딩 경계 사이의 원시 청크를 추출합니다.

    반환: [{heading, page_start, page_end, text}, ...]
    """
    # 마지막 경계를 위한 sentinel 추가
    max_page = max(page_text.keys())
    sentinel = {"page": max_page + 1, "text": "__END__", "pattern": "sentinel"}
    boundaries = headings + [sentinel]

    raw_chunks = []
    for i, h in enumerate(boundaries[:-1]):
        next_h     = boundaries[i + 1]
        h_page     = h["page"]
        next_page  = next_h["page"]

        # 현재 헤딩 페이지 ~ 다음 헤딩 직전 페이지까지 텍스트 수집
        chunk_lines = []
        for pnum in range(h_page, next_page):
            text = page_text.get(pnum, "")
            if not text:
                continue
            # 현재 페이지에서 헤딩 텍스트 이후 부분만 사용
            # (헤딩 텍스트 자체도 포함 — 문맥 유지)
            if pnum == h_page:
                idx = text.find(h["text"])
                if idx != -1:
                    text = text[idx:]  # 헤딩부터 끝까지
            chunk_lines.append(text)

        chunk_text = "\n".join(chunk_lines).strip()

        raw_chunks.append({
            "heading"    : h["text"],
            "pattern"    : h["pattern"],
            "page_start" : h_page,
            "page_end"   : next_page - 1,
            "text"       : chunk_text,
            "char_count" : len(chunk_text),
        })

    return raw_chunks


def split_by_sentence(text: str, max_chars: int) -> list[str]:
    """
    텍스트를 max_chars 이하로 문장 경계에서 분할합니다.
    문장 경계: 마침표/물음표/느낌표 뒤 공백 또는 개행.
    """
    # 문장 경계 패턴
    sentence_end = re.compile(r'(?<=[.?!])\s+')
    sentences = sentence_end.split(text)

    parts, buf = [], ""
    for sent in sentences:
        if len(buf) + len(sent) + 1 <= max_chars:
            buf = (buf + " " + sent).strip() if buf else sent
        else:
            if buf:
                parts.append(buf)
            # 단일 문장이 max_chars 초과하면 강제 분할
            if len(sent) > max_chars:
                for j in range(0, len(sent), max_chars):
                    parts.append(sent[j:j + max_chars])
                buf = ""
            else:
                buf = sent
    if buf:
        parts.append(buf)
    return [p for p in parts if p.strip()]


def apply_fallback(raw_chunks: list, min_chars: int, max_chars: int) -> list:
    """
    원시 청크에 fallback을 적용합니다.
    - 최소 미만 → 다음 청크와 병합
    - 최대 초과 → 문장 경계 분할

    반환: [{heading, page_start, page_end, text, char_count, fallback, chunk_id}, ...]
    """
    # ── 1단계: 최소 미만 병합 ─────────────────────────────────
    merged = []
    buf    = None
    for rc in raw_chunks:
        if buf is None:
            buf = rc.copy()
            buf["fallback"] = []
            continue

        if buf["char_count"] < min_chars:
            # 현재 버퍼가 너무 짧음 → 다음 청크와 병합
            buf["text"]       += "\n" + rc["text"]
            buf["char_count"]  = len(buf["text"])
            buf["page_end"]    = rc["page_end"]
            buf["heading"]     = buf["heading"] + " / " + rc["heading"]
            buf["fallback"].append("merge_min")
        else:
            merged.append(buf)
            buf = rc.copy()
            buf["fallback"] = []

    if buf:
        merged.append(buf)

    # ── 2단계: 최대 초과 분할 ─────────────────────────────────
    final  = []
    seq    = 0

    for chunk in merged:
        if chunk["char_count"] <= max_chars:
            seq += 1
            final.append({
                "chunk_id"   : seq,
                "heading"    : chunk["heading"],
                "pattern"    : chunk["pattern"],
                "page_start" : chunk["page_start"],
                "page_end"   : chunk["page_end"],
                "text"       : chunk["text"].strip(),
                "char_count" : chunk["char_count"],
                "fallback"   : chunk["fallback"] if chunk["fallback"] else None,
            })
        else:
            # 최대 초과 → 문장 경계 분할
            parts = split_by_sentence(chunk["text"], max_chars)
            for idx, part in enumerate(parts):
                seq += 1
                fb  = list(chunk["fallback"]) + ["split_max"]
                final.append({
                    "chunk_id"   : seq,
                    "heading"    : chunk["heading"] + f" ({idx+1}/{len(parts)})",
                    "pattern"    : chunk["pattern"],
                    "page_start" : chunk["page_start"],
                    "page_end"   : chunk["page_end"],
                    "text"       : part.strip(),
                    "char_count" : len(part.strip()),
                    "fallback"   : fb,
                })

    return final


print("✓ 청킹 함수 정의 완료")
print(f"  MIN_CHARS = {MIN_CHARS}자  (미만이면 다음 청크와 병합)")
print(f"  MAX_CHARS = {MAX_CHARS}자  (초과하면 문장 경계에서 분할)")


### 셀 6 — 청킹 실행
- 확인할 것: raw 청크 수와 fallback 적용 후 청크 수를 비교하세요. fallback 비율이 30% 초과하면 구조 기반 청킹의 효과가 약화됩니다.


In [ ]:
# ── 청킹 실행 ───────────────────────────────────────────────

# 1. 원시 청크 추출
raw_chunks = extract_raw_chunks(headings_final, page_text)
print(f"원시 청크 수: {len(raw_chunks)}개")

# 원시 청크 크기 분포 미리보기
rc_sizes = [c["char_count"] for c in raw_chunks]
print(f"  평균: {sum(rc_sizes)//len(rc_sizes):,}자  "
      f"최소: {min(rc_sizes):,}자  "
      f"최대: {max(rc_sizes):,}자")
print(f"  100자 미만 (병합 대상): {sum(1 for s in rc_sizes if s < MIN_CHARS)}개")
print(f"  2000자 초과 (분할 대상): {sum(1 for s in rc_sizes if s > MAX_CHARS)}개")
print()

# 2. Fallback 적용
chunks = apply_fallback(raw_chunks, MIN_CHARS, MAX_CHARS)

# fallback 통계
n_merge = sum(1 for c in chunks if c["fallback"] and "merge_min" in c["fallback"])
n_split = sum(1 for c in chunks if c["fallback"] and "split_max" in c["fallback"])
n_clean = sum(1 for c in chunks if not c["fallback"])
fb_rate = (n_merge + n_split) / len(chunks) * 100

print(f"fallback 적용 후 최종 청크: {len(chunks)}개")
print(f"  정상 청크       : {n_clean}개")
print(f"  병합(merge_min) : {n_merge}개")
print(f"  분할(split_max) : {n_split}개")
print(f"  fallback 비율   : {fb_rate:.1f}%  "
      f"{'⚠ 30% 초과 — 구조 청킹 효과 약화' if fb_rate > 30 else '✓ 양호'}")


### 셀 7 — 청크 품질 육안 확인
- 확인할 것: 크기 분포, 이상 청크(너무 짧거나 긴 것), 헤딩과 내용이 일치하는지 샘플 3개를 확인하세요.


In [ ]:
# ── 청크 품질 확인 ──────────────────────────────────────────
sizes = [c["char_count"] for c in chunks]
sorted_sizes = sorted(sizes)
n = len(sizes)

median = sorted_sizes[n // 2]
p25    = sorted_sizes[n // 4]
p75    = sorted_sizes[3 * n // 4]

print("── 크기 분포 ─────────────────────────────────────────")
print(f"  총 청크  : {n}개")
print(f"  평균     : {sum(sizes)//n:,}자")
print(f"  중앙값   : {median:,}자")
print(f"  25 pct   : {p25:,}자")
print(f"  75 pct   : {p75:,}자")
print(f"  최소     : {min(sizes):,}자")
print(f"  최대     : {max(sizes):,}자")
print()
print("  IEP-1001 최적 구간(1000~1200자) 내 청크 수:",
      sum(1 for s in sizes if 1000 <= s <= 1200))
print()

# 이상 청크 출력
tiny  = [c for c in chunks if c["char_count"] < MIN_CHARS]
giant = [c for c in chunks if c["char_count"] > MAX_CHARS]
print(f"── 이상 청크 ({MIN_CHARS}자 미만 또는 {MAX_CHARS}자 초과) ──────────")
if tiny:
    print(f"  ⚠ {MIN_CHARS}자 미만 {len(tiny)}개:")
    for c in tiny:
        print(f"    [chunk {c['chunk_id']}] p.{c['page_start']}~{c['page_end']}  {c['char_count']}자  {c['heading'][:50]}")
if giant:
    print(f"  ⚠ {MAX_CHARS}자 초과 {len(giant)}개:")
    for c in giant:
        print(f"    [chunk {c['chunk_id']}] p.{c['page_start']}~{c['page_end']}  {c['char_count']}자  {c['heading'][:50]}")
if not tiny and not giant:
    print(f"  ✓ 이상 청크 없음")
print()

# 샘플 청크 3개 출력 (앞 / 중간 / 뒤)
samples = [chunks[0], chunks[len(chunks)//2], chunks[-1]]
print("── 샘플 청크 (앞 / 중간 / 뒤) ───────────────────────")
for c in samples:
    print(f"\n  [chunk {c['chunk_id']}] p.{c['page_start']}~{c['page_end']}  {c['char_count']}자  fallback={c['fallback']}")
    print(f"  헤딩: {c['heading'][:70]}")
    print(f"  내용: {c['text'][:200].replace(chr(10), ' ')}...")


### 셀 8 — 크기 분포 히스토그램
- 확인할 것: 분포 모양이 IEP-1001 최적 구간(1000~1200자) 근방에 집중되어 있는지, 혹은 어느 구간에 몰려 있는지 확인하세요.


In [ ]:
# ── 크기 분포 히스토그램 (matplotlib) ──────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bins = list(range(0, max(sizes) + 300, 200))

# 왼쪽: 전체 분포
ax = axes[0]
ax.hist(sizes, bins=bins, color="#4a90d9", edgecolor="white", linewidth=0.5)
ax.axvline(1000, color="#e67e22", linewidth=1.5, linestyle="--", label="IEP-1001 최적 하한 (1000자)")
ax.axvline(1200, color="#e74c3c", linewidth=1.5, linestyle="--", label="IEP-1001 최적 상한 (1200자)")
ax.set_xlabel("청크 크기 (자)")
ax.set_ylabel("청크 수")
ax.set_title("구조 기반 청킹 — 청크 크기 분포 (전체)")
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# 오른쪽: 0~3000자 구간 확대
ax2 = axes[1]
ax2.hist([s for s in sizes if s <= 3000], bins=range(0, 3200, 200),
         color="#5dade2", edgecolor="white", linewidth=0.5)
ax2.axvline(1000, color="#e67e22", linewidth=1.5, linestyle="--")
ax2.axvline(1200, color="#e74c3c", linewidth=1.5, linestyle="--")
ax2.set_xlabel("청크 크기 (자)")
ax2.set_ylabel("청크 수")
ax2.set_title("0~3000자 구간 확대")
ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()

chart_path = f"{OUTPUT_DIR}/chunk_dist_structural.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ 저장: {chart_path}")


### 셀 9 — JSON 저장 및 Step 2 완료 선언
- 확인할 것: `ipcc_chunks_structural.json`이 생성됐는지, 청크 수와 파일 크기가 출력되는지.


In [ ]:
# ── 청크 JSON 저장 ──────────────────────────────────────────
chunks_path = f"{OUTPUT_DIR}/ipcc_chunks_structural.json"
with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

file_kb = Path(chunks_path).stat().st_size / 1024

# 최종 헤딩 리스트도 갱신 저장
final_headings_path = f"{OUTPUT_DIR}/ipcc_headings_final.json"
with open(final_headings_path, "w", encoding="utf-8") as f:
    json.dump(headings_final, f, ensure_ascii=False, indent=2)

print(f"""
╔══════════════════════════════════════════════════╗
  Day 2 — Step 2 청킹 완료 요약
╚══════════════════════════════════════════════════╝

  최종 헤딩 수    : {len(headings_final)}개
  원시 청크       : {len(raw_chunks)}개
  최종 청크       : {len(chunks)}개
  fallback 비율   : {fb_rate:.1f}%
    병합(merge_min): {n_merge}개
    분할(split_max): {n_split}개

  청크 크기
    평균    : {sum(sizes)//len(sizes):,}자
    중앙값  : {median:,}자
    최소    : {min(sizes):,}자
    최대    : {max(sizes):,}자
  IEP-1001 최적 구간(1000~1200자) 내: {sum(1 for s in sizes if 1000<=s<=1200)}개

  저장 파일:
    ✓ ipcc_chunks_structural.json  ({len(chunks)}청크, {file_kb:.0f} KB)
    ✓ ipcc_headings_final.json     ({len(headings_final)}개)
    ✓ chunk_dist_structural.png    (크기 분포 히스토그램)

  다음 단계: Step 3 — ChromaDB 벡터 저장 및 스모크 테스트
""")
